## Prepare the kernel (Python 3) to run with the libraries versions

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.0.3</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>

In this notebook we set up all libraries needed for the labs

In [ ]:
# Bootstrap the installers: keep pip current and install uv
# (fast resolver/installer used throughout these labs).
!pip install -q -U pip uv

In [ ]:
# Install the full shared kernel stack from the repo-root requirements.txt
# using uv. This requirements.txt is the SINGLE SOURCE OF TRUTH for the
# notebook-kernel dependencies used across all labs. We remove any
# pre-existing SageMaker SDK packages first for a clean install.
import sys
!uv pip uninstall --python {sys.executable} sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops || true
!uv pip install --python {sys.executable} -r requirements.txt

## Install SageMaker Python SDK `3.13.1`

In [ ]:
# Restart kernel to pick up the updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Imports checks

If the imports works, we are good to go. Otherwise you need to restart the kernel to pick up the updated packages

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve

import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

## Install MLflow and the sagemaker-mlflow plugin for authentication

In [ ]:
# MLflow and the sagemaker-mlflow plugin are included in the root
# requirements.txt (installed above) - no separate install needed.

In [ ]:
# pandas, matplotlib and awswrangler are included in the root
# requirements.txt (installed above) - no separate install needed.

In [ ]:
# torchvision is included in the root requirements.txt
# (installed above) - no separate install needed.

## Connect to MLflow app server

MLflow is an open-source platform for managing the ML lifecycle, including experiment tracking, model versioning, and deployment. SageMaker AI provides managed MLFlow Apps that integrate seamlessly with your ML workflows.

Note: MLflow Apps are the latest managed MLflow offering on SageMaker AI and should be preferred over existing MLflow Tracking Servers. MLflow Apps offer additional features such as:

* Serverless capability that eliminates infrastructure management
* Faster startup time (~2 minutes)
* Cross-account sharing via AWS RAM
* Automatic in-place version upgrades
* Integration with SageMaker Pipelines

Let's start with an overview of MLFlow tracking features:

* Organize Experiments: MLflow structures experimentation with experiments that contain multiple runs. Each run captures parameters, metrics, and artifacts from a single execution of your ML code. You can picture experiments as the top level folder for organizing your hypotheses, and runs as individual test executions within that experiment.
* Track Experiments: MLFlow allows you to track experiments by logging parameters, metrics, and artifacts using the MLFlow API. You can use `mlflow.log_param()`, `mlflow.log_metric()`, and `mlflow.log_artifact()` to capture experiment data.
* Compare and Evaluate Experiments: The MLFlow UI makes it easy to compare different runs, visualize metrics, and identify the best combination of hyperparameters.

If you're running an AWS-led workshop or used the delivered CloudFormation template to provision your workshop environment, an MLflow app server must be up and running. If you don't have an MLflow app server, follow the Developer Guide or run the following code cell to create a new one. To create and manage an MLflow app server and to work with managed MLflow experiements, you need the following permissions attached to the SageMaker execution role:

```json
{
    "Version":"2012-10-17",		 	 	     
    "Statement": [        
        {            
            "Effect": "Allow",            
            "Action": [
                "sagemaker-mlflow:*",
                "sagemaker:CreateMlflowApp",
                "sagemaker:ListMlflowApps",
                "sagemaker:UpdateMlflowApp",
                "sagemaker:DeleteMlflowApp",
                "sagemaker:StartMlflowApp",
                "sagemaker:StopMlflowApp",
                "sagemaker:CreatePresignedMlflowAppUrl"
            ],            
            "Resource": "*"        
        }        
    ]
}
```
and you need the following permissions attached to the MLflow server app itself:
```json
{
    "Version":"2012-10-17",		 	 	 
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:Get*",
                "s3:Put*",
                "s3:List*",
                "sagemaker:AddTags",
                "sagemaker:CreateModelPackageGroup",
                "sagemaker:CreateModelPackage",
                "sagemaker:UpdateModelPackage",
                "sagemaker:DescribeModelPackageGroup"
            ],
            "Resource": "*"
        }
    ]
}
```
Execute the following code to check if you have a running MLflow server app.

In [ ]:
import boto3
import numpy as np
import pandas as pd
import os
import json

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from IPython.display import Javascript, HTML

boto_session = boto3.Session()
sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name
sm_client = boto_session.client("sagemaker")

In [ ]:
import mlflow
import time
from datetime import date

suffix = date.today().isoformat()

mlflow_name = 'mlflow-app'
bucket_name = Session().default_bucket()

# Check for existing MLflow Apps
apps = sm_client.list_mlflow_apps().get('Summaries', [])
mlflow_app = next((a for a in apps if a['Name'] == mlflow_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app['Arn'])
else:
    # Create new MLflow App
    print(f"Creating MLflow App: {mlflow_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_name,
        ArtifactStoreUri=f's3://{bucket_name}',
        RoleArn=role,
        ModelRegistrationMode='AutoModelRegistrationEnabled' # enables autoregistration of models from MLflow registry to SageMaker registry
    )
    # Wait for app to be ready
    while True:
        mlflow_app = sm_client.describe_mlflow_app(Arn=response['Arn'])
        if mlflow_app['Status'] in ['Created', 'Updated']:
            break
        elif mlflow_app['Status'] in ['CreateFailed', 'Deleted']:
            raise RuntimeError(f"MLflow App creation failed: {mlflow_app['Status']}")
        print(f"Status: {mlflow_app['Status']}... waiting")
        time.sleep(30)

# Wait if app is still being created
while mlflow_app['Status'] in ['Creating', 'Updating']:
    print(f"MLflow App creating... waiting")
    time.sleep(30)
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app['Arn'])

mlflow_arn = mlflow_app['Arn']
print(f"MLflow App: {mlflow_app['Name']} (v{mlflow_app.get('MlflowVersion', 'N/A')})")
print(f"ARN: {mlflow_arn}")